In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

import httpx
import roboflow


def list_files(path):
    for entry in os.scandir(path):
        if entry.is_dir():
            yield from list_files(entry.path)
        else:
            yield entry


roboflow.login()
rf = roboflow.Roboflow()

workspace = rf.workspace("agrowizard")
project = workspace.project("stick-detection")

In [ ]:
def exact_filename_lookup(filename: str, split: str):
    url = "https://api.roboflow.com/agrowizard/search/v1"
    r = httpx.post(
        url,
        params={"api_key": rf.api_key},
        json={"query": f"filename:{filename} AND split:{split}", "pageSize": 10, "fields": ["id", "filename"]},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    return [item for item in data.get("results", []) if item.get("filename") == filename]

In [ ]:
def add_tag(image_id: str, tag: str):
    url = f"https://api.roboflow.com/agrowizard/stick-detection/images/{image_id}/tags"
    r = httpx.post(
        url,
        params={"api_key": rf.api_key},
        json={"operation": "add", "tags": [tag]},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

In [ ]:
for split in ("valid", "test", "train"):
    for direction in ("false_positives", "false_positives"):
        for typ in ("sticks", "poles", "trees"):
            directory = f"runs/checkpoint_best_ema_{split}_evaluation/{direction}/{typ}"
            print(f"Checking directory: {directory}")

            files = set()
            for file in list_files(directory):
                name, ext = tuple(file.name.split(".rf.")[0].split("_"))
                files.add(f"{name}.{ext}")

            with ThreadPoolExecutor(max_workers=10) as executor:
                found_images = list(executor.map(lambda filename: exact_filename_lookup(filename, split), files))

            found_images = [x for xs in found_images for x in xs]
            print(
                f"Found {len(found_images)} images on Roboflow (vs {len(files)} locally), tagging with {typ}-{direction.replace('_', '-')}"
            )

            # with ThreadPoolExecutor(max_workers=10) as executor:
            #     list(
            #         executor.map(
            #             lambda image: add_tag(image["id"], f"{typ}-{direction.replace('_', '-')}"),
            #             found_images,
            #         )
            #     )